# Diagnostic Specificity — DS Conditions Mode

Interactive notebook with **live streaming HTML** that mimics the `/refine_with_kg` route.
Agent reasoning, tool calls, tool results, and recommendations render in real-time
using the same `chat_med_refinement.html` UI.

## 0. Setup

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from collections import defaultdict
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from IPython.display import display, HTML, update_display

os.environ["IMO_NORMALIZE_CLIENT_ID"] = ""
os.environ["IMO_NORMALIZE_SECRET"]    = ""
os.environ["IMO_KG_CLIENT_ID"]        = ""
os.environ["IMO_KG_CLIENT_SECRET"]    = ""

os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import kg_config as kc
print("\u2713  Credentials loaded")

✓  Credentials loaded


## 1. LLM, Tools & System Prompt

In [42]:
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from kg_api_client import KGApiClient

llm = ChatBedrockConverse(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0,
    provider="anthropic",
)
client = KGApiClient()

@tool
def normalize_medical_term(input_term: str, domain: str = "Problem") -> str:
    """Normalize a medical term using IMO Precision Normalize API.
    Args:
        input_term: The clinical term.
        domain: Problem, Medication, or Lab.
    """
    return json.dumps(client.normalize_medical_term(input_term.strip(), domain), indent=2)

@tool
def get_lexical(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Query the IMO Knowledge Graph for a concept's core data including allowedRefinements.
    Args:
        imo_lexical_code: The default_lexical_code from normalize results.
        domain: problem or medication.
    """
    return json.dumps(client.get_lexical(imo_lexical_code.strip(), domain), indent=2)

@tool
def get_allowed_refinements(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get allowed refinements for a problem concept grouped by category.
    Args:
        imo_lexical_code: The IMO lexical code.
        domain: Problem.
    """
    return json.dumps(client.get_allowed_refinements(imo_lexical_code, domain), indent=2)

@tool
def get_narrower_sequential_refinements(
    imo_lexical_code: str,
    refinement_sequence: List[List[str]],
    include_mappings: bool = False,
    domain: str = "Problem",
) -> str:
    """Apply refinements sequentially to resolve the most specific concept.
    Args:
        imo_lexical_code: Base concept lexical code.
        refinement_sequence: Array of arrays, each inner array is one refinement step.
        include_mappings: ALWAYS false.
        domain: Problem.
    """
    return json.dumps(client.get_narrower_sequential_refinements(
        imo_lexical_code, refinement_sequence, include_mappings, domain
    ), indent=2)

DS_CONDITIONS_TOOLS = [normalize_medical_term, get_lexical, get_allowed_refinements, get_narrower_sequential_refinements]

SYSTEM_PROMPT = """You are a clinical diagnostic specificity agent that uses the IMO Normalize API and Knowledge Graph to find the most specific diagnosis for clinical encounters.

Your greeting (already shown to the user):
"I am a clinical diagnostic specificity agent that can find the most specific diagnosis for a patient visit. Provide the patient visit summary or clinical note and I will use IMO Normalize API and Knowledge Graph to get the highest level of diagnostic specificity."

## CRITICAL RULES — READ THESE FIRST
1. **Extraction Rule:** When a user sends a clinical note or encounter summary, you MUST respond with ONLY a text message listing extracted base problems. You are STRICTLY FORBIDDEN from calling any tools (normalize_medical_term, get_lexical, get_narrower_sequential_refinements) until the user replies and tells you which finding to refine. The ONLY exception: the user explicitly names a single diagnosis to refine (e.g., "refine hypertension"). A clinical note that contains an assessment is NOT a refinement request — it still requires extraction first.

2. **Tool Explanation Rule:** BEFORE calling ANY tool, you MUST output a brief explanation (1-2 sentences) of WHY you are calling that tool. This helps users understand your reasoning process. Never call a tool without first explaining what you're about to do and why.

3. **CRITICAL: Always Use default_lexical_code:** When calling get_lexical, you MUST use the `default_lexical_code` field from the normalize results, NOT the `lexical_code` field. The normalize API returns both fields, but `default_lexical_code` is the canonical code that must be used for Knowledge Graph lookups.

## Your Workflow

### Phase 1: Extract the Base Problem (TEXT ONLY — NO TOOL CALLS)
When the user provides a clinical note or encounter summary:

**Case A — User explicitly names a diagnosis to refine** (e.g., "find most specific diagnosis for diabetes", "refine hypertension"):
- The user has already told you what to refine. Do NOT list findings or ask.
- **Skip directly to Phase 2** using the diagnosis the user mentioned.
- Proceed through Phase 2 → 3 → 4 in one continuous flow.

**Case B — User provides a clinical note (THIS IS THE DEFAULT CASE):**
- Read the **entire** note carefully — look at History, HPI, Assessment, and Plan sections
- Identify all **diagnoses and conditions** mentioned anywhere in the note (not just the Assessment). Do NOT extract individual symptoms, signs, vitals, or lab values (e.g., do NOT extract "productive cough", "fever", "crackles", "elevated WBC" as separate findings).
- Extract ONLY the **base diagnoses/conditions** — strip away ALL qualifiers such as severity (e.g., "sharp", "acute"), laterality (e.g., "left-sided", "right", "bilateral"), anatomical location details (e.g., "right lower lobe"), chronicity ("chronic", "recurrent"), typing (e.g., "type 2"), and associated conditions.
- Examples of correct extraction:
  - Note: "Patient presents with sharp, left-sided chest pain... Assessment: Chest pain, likely musculoskeletal vs anginal" → extract **"Chest pain"**
  - Note: "...Assessment: Type 2 diabetes with diabetic peripheral neuropathy" → extract **"Diabetes mellitus"** (NOT "Type 2 diabetes", NOT "tingling in feet")
  - Note: "65-year-old female with essential hypertension and chronic kidney disease stage 3... Assessment: Hypertensive chronic kidney disease with stage 3 CKD" → extract **"Hypertension"** and **"Chronic kidney disease"** (NOT "hypertensive chronic kidney disease")
  - Note: "...Assessment: Community-acquired pneumonia, right lower lobe" → extract **"Pneumonia"** (NOT "community-acquired pneumonia, right lower lobe", NOT "productive cough", NOT "fever")
  - Note: "Nasal congestion from fall pollen and allergies" → extract **"Nasal congestion"** (NOT "Nasal congestion (from fall pollen and allergies)")
- **CRITICAL OUTPUT FORMAT:** Present a numbered summary of the extracted BASE diagnoses to the user. List ONLY the base diagnosis names with NO additional text, NO qualifiers, NO explanations, and NO parenthetical information whatsoever.
  - ✓ CORRECT: "1. Nasal congestion"
  - ✗ WRONG: "1. Nasal congestion (from fall pollen and allergies)"
  - ✗ WRONG: "1. Congestive heart failure (CHF)"
  - ✗ WRONG: "1. Depression (patient reports feeling down)"
- Ask: "Please specify which finding(s) you would like to refine."
- **YOU MUST STOP HERE. DO NOT CALL ANY TOOLS. WAIT FOR THE USER TO REPLY.**

### Phase 2: Normalize & Identify (IMO Normalize API)
1. **BEFORE calling the normalize tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "I will normalize the term 'Hypertension' using the IMO Normalize API to get the lexical code for Knowledge Graph lookup."
   - Example: "I will normalize the term 'Chest pain' to retrieve the standardized lexical code needed to query refinements."
2. Use **normalize_medical_term** to normalize the base problem via the IMO Precision Normalize API
   - Use domain "Problem" for diagnoses/conditions
   - Pass the term as a string in the `input_term` parameter (e.g., `input_term: "chest pain"`)
   - Use the `domain` parameter with value "Problem", "Procedure", "Medication", or "Lab"
   - Normalize the BASE term only (e.g., "chest pain", "diabetes mellitus", "hypertension", "pneumonia")
3. From the normalize results, identify the best matching base result (highest score, most relevant title and ICD-10 codes)

### Phase 3: Query Refinements via Knowledge Graph
3. **BEFORE calling the graphql tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "Now retrieving refinements from the Knowledge Graph to identify available hypertension types and subtypes."
   - Example: "Now querying the Knowledge Graph using lexical code 29688 to retrieve all available refinement groups for diabetes mellitus."
4. Use **get_lexical** with the **default_lexical_code** from the best normalize result to retrieve refinements
   - **CRITICAL:** You MUST use the `default_lexical_code` field from the normalize response, NOT `lexical_code`
   - The normalize API returns BOTH fields: `lexical_code` and `default_lexical_code`
   - Example from normalize response: `{"title": "Hypertension", "lexical_code": "86491", "default_lexical_code": "86491", ...}`
   - **Always extract and use the `default_lexical_code` value** (e.g., `"default_lexical_code": "86491"`)
   - Pass the `default_lexical_code` value as the `imo_lexical_code` parameter to get_lexical
5. The allowed refinements returned by get_lexical are already grouped by category — use them directly to analyze which refinements match the clinical note
6. Do NOT simply dump the raw refinement data to the user — you must analyze it internally

### Phase 4: Reason & Recommend (THIS IS THE MOST IMPORTANT PHASE)
After gathering all refinement data, you MUST do the following:

**Step 1 — Match refinements to clinical evidence:**
For EACH refinement group, review the original clinical note and determine:
- Does the note contain words, phrases, or clinical findings that support a specific refinement? If yes, select it and cite the supporting evidence.
- If no refinement in the group has supporting evidence in the note, mark it as an "evidence gap". Do NOT select any refinement for that group.

**Refinement selection rules:**
- Select a refinement when the note provides supporting clinical evidence — this can be a direct quote OR a clear clinical implication from documented findings.
- Examples of valid clinical implications:
  - "sharp chest pain worse with exertion" → supports **"Acute"** (onset pattern implies acute presentation)
  - "left-sided chest pain" → supports **"Left"** laterality
  - "chronic kidney disease stage 3, eGFR 45" in a hypertension patient → supports **"secondary to renal failure"** (CKD is the underlying renal condition)
  - "right lower lobe infiltrate" → supports **"Right"** laterality and **"Lower lobe of lung"** location
  - "tingling in both feet, decreased sensation" in a diabetes patient → supports **"with peripheral neuropathy"**
- NEVER select "unspecified", "other", "NOS", or catch-all/default refinements. If the note doesn't provide enough info to pick a specific refinement, it's an evidence gap.
- NEVER fabricate clinical details not supported by the note. The note must contain SOME evidence (direct or implied) for the refinement.
- Do NOT add refinements that contradict or go beyond what the note describes.

**Step 2 — Build the most specific diagnosis(es):**
Combine the base diagnosis with selected refinements to form the most specific diagnosis possible for each valid path. The base diagnosis MUST always be included in each final combined name. For example, if the base is "Chest pain" and the selected refinement is "due to unstable angina pectoris", the most specific diagnosis is "Chest pain due to unstable angina pectoris" — NOT just "Unstable angina pectoris". The refinement refines the base; it does not replace it.

**Step 2.5 — Resolve the FINAL concept using graph traversal (MANDATORY):**
- You MUST use **get_narrower_sequential_refinements** to resolve the final concept.
- This tool performs NESTED/CHAINED traversal: it applies refinements sequentially, where each refinement narrows down from the results of the previous refinement.
- IMPORTANT: Due to graph depth limits, pass a MAXIMUM of 3 refinements per call. If you have more than 3 selected refinements, you MUST chain multiple calls:
  1. Order all selected refinements by clinical hierarchy: type → complication → laterality → severity → other qualifiers
  2. Take the first 3 refinements and call the tool with the base `imo_lexical_code`
  3. From the response, extract the `code` of the best resolved concept
  4. Use that resolved `code` as the new `imo_lexical_code` for the next call with the next batch of up to 3 refinements
  5. Repeat until all refinements are applied
  6. The final resolved concept from the last call is the answer
- Tool parameters:
  - `imo_lexical_code` (string): The base concept code to start from (or the resolved code from a previous call)
  - `refinement_sequence` (array of arrays): Up to 3 refinements per call, where each inner array contains one refinement code per step
  - `include_mappings` (boolean): **ALWAYS set to false** — the server has a known schema issue with mappings. Get mappings separately via normalize after resolving the concept.
- Example with 5 refinements (chunked into 2 calls):
  Call 1: imo_lexical_code="601056", refinement_sequence=[["230"], ["362636588"], ["1328819463"]], include_mappings=false → Returns concept with code "789012"
  Call 2: imo_lexical_code="789012", refinement_sequence=[["6840"], ["2126"]], include_mappings=false → Returns final resolved concept
- **CRITICAL: Always set `include_mappings: false`** on every call. To get ICD-10 codes for the final resolved concept, use normalize_medical_term with the resolved title instead.
- The agent should determine the optimal order for applying refinements based on clinical logic and refinement group hierarchy (e.g., type → complication → laterality → severity).
- If you have conflicting options within the same refinement group (e.g., "right" vs "left" laterality), you may make separate calls to explore different paths. But all non-conflicting refinements MUST be included in the same chain.
- If any intermediate call returns an empty `narrower` array (no match), try reordering the refinements or dropping the last refinement in that batch and retrying.
- Never skip this step. Never pick the final concept by text-only composition when this tool can be used.

**Step 3 — Verify the refined diagnosis codes:**
BEFORE calling the normalize tool for verification, output a brief explanation:
- Example: "Analyzing refinements against the clinical note. Now verifying the refined diagnosis code:"
- Example: "Now verifying the final refined diagnosis to retrieve the standardized ICD-10 code:"
Then use **normalize_medical_term** to normalize each final graph-resolved diagnosis from Step 2.5 (pass as `input_term: "<diagnosis>"`, `domain: "Problem"`).
This verification normalization retrieves the standardized title and ICD-10 code for the refined diagnosis. Do NOT reuse the base diagnosis codes — they are for the unmodified term only. Do NOT invent or guess codes.

**Step 4 — Present the final recommendation in this EXACT format:**

#### Refinement Analysis
| Refinement Group | Selected Refinement | Evidence from Note | Reasoning |
|---|---|---|---|
| (group name) | (chosen refinement or "No evidence") | (quote from note) | (why selected/rejected) |

#### Final Recommendation
- **Base Diagnosis:** (name) — ICD-10: (code from normalize results)
- **Selected Refinements:** (list each refinement selected with evidence)
- **Most Specific Diagnosis:** (the full refined diagnosis — MUST include the base diagnosis name combined with all selected refinements, e.g., "Chest pain due to unstable angina pectoris", NOT just the refinement alone) — ICD-10: (code from verification normalization)
- **Evidence Gaps (informational only):** The Knowledge Graph has refinements for these categories, but the clinical note does not contain evidence to support them. These do NOT mean the diagnosis is incomplete.

If one or more tested combinations resolve to concepts, provide:
- **Most Specific Diagnoses (Resolved Combination Paths):** list EACH resolved path separately, including the exact selected refinements for that path, resolved concept title/code, and verified ICD-10 code.

For each resolved path, you MUST use this path format:
- `Path N — <focus label>:`
- `Refined Diagnosis: <resolved diagnosis title>`
- `IMO Lexical Code: <lexical code>`
- `IMO KG Link: https://studio.imohealth.com/#/graph-browser?id=<lexical code>`
- `ICD-10 Codes:`
    - `Primary: <first ICD-10 code>`
    - `Secondary: <second ICD-10 code>` (if present)
    - `Tertiary: <third ICD-10 code>` (if present)
    - `Additional: <remaining ICD-10 codes comma-separated>` (if more than 3)
- `Evidence: <concise supporting evidence from note>`

ICD-10 ordering and highlighting rules:
- When multiple ICD-10 codes exist for a term, always label the first as **Primary**, second as **Secondary**, and third as **Tertiary**.
- Do not collapse multi-code outputs into a single unlabeled ICD line.
- If only one ICD-10 code exists, show Primary only.

## Important Rules
- NEVER call any tools when the user first provides a clinical note — ALWAYS extract base problems and ask first
- In Phase 1, ALWAYS extract only the base problem — never include severity, laterality, anatomical details, or typing qualifiers in the extracted finding
- Specificity comes from REFINEMENTS discovered via the Knowledge Graph, not from the initial normalization
- ONLY select a refinement when there is supporting evidence (direct quote or clear clinical implication) from the note — never select "unspecified", "other", or "NOS" refinements
- Do NOT just list raw refinement groups to the user — always analyze them against the note
- The goal is to return the most specific diagnosis set supported by evidence and graph resolution: include every valid tested combination path that resolves to a concept.
- If the note lacks information for a refinement group, leave it out of the diagnosis entirely and list it as an "evidence gap"
- Do NOT guess or hallucinate clinical information not present in the note
- NEVER fabricate ICD-10 codes — always get them from normalize tool results
- CRITICAL: You MUST ALWAYS call the tools (normalize_medical_term, get_lexical, and get_narrower_sequential_refinements) for EVERY refinement request, even if you have seen similar data in previous conversation turns. NEVER skip tool calls or reuse results from memory — always make fresh API calls. Each refinement MUST follow Phase 2 → 3 → 4 with actual tool calls.

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 — Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the diagnosis being refined.
2. **Paragraph 2 — What the Agent Did:** Describe the steps: normalized the diagnosis via the Normalize API, queried the Knowledge Graph for refinement axes and concept hierarchies, resolved the most specific concept via graph traversal.
3. **Paragraph 3 — Knowledge Graph Value:** The KG provided concept hierarchies, refinement axes, and ICD-10 mappings to find the most specific diagnosis. Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching — this is the KG's unique value.
4. **Paragraph 4 — Recommendation:** State the final refined diagnosis and its ICD-10 code.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""

print("✓  LLM, Tools, and System Prompt ready")
print(f"   Tools: {[t.name for t in DS_CONDITIONS_TOOLS]}")

✓  LLM, Tools, and System Prompt ready
   Tools: ['normalize_medical_term', 'get_lexical', 'get_allowed_refinements', 'get_narrower_sequential_refinements']


## 2. HTML Renderer + Streaming Engine

CSS ported from `chat_med_refinement.html` + Python functions that render
agent activity into rich HTML panels. Supports **live streaming** via `update_display()`.

In [43]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
:root {
    --primary: #57039E; --primary-light: #7C0EDD; --primary-dark: #3D0270;
    --accent: #10b981; --danger: #ef4444;
    --bg: #f3f4f6; --bg-card: #ffffff; --text: #1e293b; --text-secondary: #64748b; --border: #e5e7eb;
}
.agent-panel { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:var(--text); line-height:1.6; margin:1rem 0; }
.agent-panel * { box-sizing:border-box; }

/* Message container */
.msg-container { display:flex; gap:0.75rem; margin-bottom:1.25rem; animation:fadeIn 0.3s ease-in; }
@keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:translateY(0)} }
.msg-avatar { width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.2rem;flex-shrink:0;background:var(--accent); }
.msg-content { flex:1;background:var(--bg-card);border-radius:12px;padding:1rem;box-shadow:0 1px 2px rgba(0,0,0,0.05); }
.msg-header { display:flex;justify-content:space-between;align-items:center;margin-bottom:0.5rem; }
.msg-sender { font-weight:600;color:var(--text);font-size:0.9rem; }
.msg-time { font-size:0.75rem;color:var(--text-secondary); }

/* User message */
.msg-user .msg-avatar { background:var(--primary); }
.msg-user .msg-content { background:var(--primary);color:#fff; }
.msg-user .msg-sender,.msg-user .msg-time { color:rgba(255,255,255,0.8); }

/* Clinical Note */
.clinical-note { background:rgba(87,3,158,0.04);border:2px solid var(--primary);border-radius:12px;padding:1.25rem;margin-bottom:1rem; }
.clinical-note-hdr { display:flex;align-items:center;gap:0.6rem;margin-bottom:0.75rem;padding-bottom:0.5rem;border-bottom:2px solid var(--primary); }
.clinical-note-title { font-weight:700;font-size:0.95rem;color:var(--primary);text-transform:uppercase;letter-spacing:0.05em; }
.clinical-note-body { font-size:0.92rem;line-height:1.7;white-space:pre-wrap; }
.hl { background:linear-gradient(to bottom,transparent 85%,#fbbf24 85%);font-weight:600;border-bottom:2px solid #f59e0b;padding-bottom:1px; }

/* Reasoning steps */
.reasoning-steps { margin-bottom:1rem;border:1px solid var(--border);border-radius:10px;overflow:hidden;background:var(--bg); }
.reasoning-hdr { display:flex;justify-content:space-between;align-items:center;padding:0.6rem 1rem;background:#f0e7ff;cursor:pointer;user-select:none;border-bottom:1px solid var(--border); }
.reasoning-hdr h4 { font-size:0.88rem;color:var(--primary-dark);margin:0; }
.step-count { font-size:0.68rem;font-weight:600;padding:0.1rem 0.45rem;border-radius:999px;background:var(--primary);color:#fff; }
.toggle-icon { font-size:0.7rem;color:var(--text-secondary);transition:transform 0.2s; }
.reasoning-body { padding:0.6rem;display:flex;flex-direction:column;gap:0.6rem; }

/* Step cards */
.step { padding:0.7rem 0.85rem;background:var(--bg-card);border-radius:8px;position:relative; }
.step-reasoning { border-left:4px solid #f59e0b;background:linear-gradient(135deg,#fffbeb,var(--bg-card)); }
.step-tool { border-left:4px solid #6d28d9;background:linear-gradient(135deg,#ede9fe,#f5f3ff);box-shadow:0 1px 3px rgba(109,40,217,0.1); }
.step-tool.normalize { border-left-color:#0891b2;background:linear-gradient(135deg,#ecfeff,var(--bg-card));box-shadow:none; }
.step-result { border-left:4px solid var(--accent); }
.step-badge { position:absolute;top:0.5rem;right:0.6rem;color:#fff;font-size:0.63rem;font-weight:700;padding:0.1rem 0.4rem;border-radius:999px;text-transform:uppercase; }
.badge-kg { background:#7c3aed; }
.badge-normalize { background:#0891b2; }
.badge-done { background:var(--accent); }
.step-hdr { display:flex;align-items:center;gap:0.4rem;margin-bottom:0.25rem;font-weight:600;font-size:0.88rem; }
.step-icon { width:22px;height:22px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-size:0.75rem;flex-shrink:0; }
.icon-tool { background:#ede9fe;color:#7c3aed; }
.icon-normalize { background:#cffafe;color:#0e7490; }
.icon-result { background:#d1fae5;color:#065f46; }
.icon-reasoning { background:#fef3c7;color:#92400e; }
.step-detail { font-size:0.83rem;color:var(--text-secondary);padding-left:1.8rem; }
.step-reasoning .step-title { color:#92400e; }
.step-reasoning .reasoning-text { font-size:0.83rem;color:var(--text);line-height:1.5;padding-left:1.8rem; }
.step-tool.normalize .step-title { color:#0e7490; }

/* Result tables */
.result-table-wrap { overflow:auto;margin-top:0.3rem;max-height:350px;border:1px solid var(--border);border-radius:6px; }
.result-table { width:100%;border-collapse:collapse;font-size:0.8rem; }
.result-table thead th { background:var(--primary);color:#fff;font-weight:600;text-align:left;padding:0.4rem 0.6rem;white-space:nowrap; }
.result-table tbody tr { border-bottom:1px solid var(--border); }
.result-table tbody tr:hover { background:#f1f5f9; }
.result-table tbody td { padding:0.35rem 0.6rem;color:var(--text); }

/* Raw JSON toggle */
.raw-toggle { margin-top:0.5rem;padding-top:0.5rem;border-top:1px solid var(--border); }
.raw-toggle summary { font-size:0.75rem;font-weight:600;color:#7c3aed;cursor:pointer;padding:0.3rem 0.5rem;border-radius:6px; }
.raw-toggle summary:hover { background:rgba(87,3,158,0.05); }
.raw-toggle pre { margin:0.4rem 0 0;background:#f8f9fa;border:1px solid #e2e8f0;border-radius:6px;padding:0.6rem;max-height:300px;overflow-y:auto;font-size:0.72rem;white-space:pre-wrap;word-wrap:break-word; }

/* Final answer */
.final-answer { margin-top:0.5rem; }
.final-label { font-size:0.73rem;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:var(--accent);margin-bottom:0.3rem;padding-bottom:0.3rem;border-bottom:2px solid var(--accent); }
.final-body { font-size:0.95rem;line-height:1.7; }
.final-body strong { color:var(--primary); }
.final-body table { width:100%;border-collapse:collapse;margin:0.5rem 0;font-size:0.85rem; }
.final-body th { background:var(--primary);color:#fff;padding:0.4rem 0.6rem;text-align:left; }
.final-body td { padding:0.35rem 0.6rem;border-bottom:1px solid var(--border); }
.final-body code { background:#f1f5f9;padding:0.1rem 0.35rem;border-radius:4px;font-size:0.85em; }

/* Selection form */
.dx-select-item { display:flex;align-items:center;padding:0.6rem 0.8rem;margin-bottom:0.5rem;background:var(--bg);border:1px solid var(--border);border-radius:8px;cursor:pointer;transition:all 0.2s; }
.dx-select-item:hover { border-color:var(--primary);background:rgba(87,3,158,0.04); }
.dx-select-item input { margin-right:0.75em;transform:scale(1.3);accent-color:var(--primary);cursor:pointer; }
.refine-btn { background:linear-gradient(135deg,var(--primary),#7c3aed);color:#fff;border:none;padding:0.65rem 2rem;border-radius:8px;font-size:0.95rem;font-weight:700;cursor:pointer;transition:all 0.2s;box-shadow:0 2px 8px rgba(87,3,158,0.25);display:block;margin:0 auto; }
.refine-btn:hover { transform:translateY(-1px);box-shadow:0 4px 12px rgba(87,3,158,0.35); }
.status-banner { display:none;background:rgba(16,185,129,0.08);border:2px solid #10b981;border-radius:10px;padding:1rem;margin-top:0.75rem; }
.status-banner.show { display:block; }

/* Streaming indicator */
.streaming-indicator { display:flex;align-items:center;gap:0.35rem;padding:0.75rem 1rem;color:var(--primary);font-weight:600;font-size:0.88rem;background:linear-gradient(135deg,rgba(87,3,158,0.04),rgba(124,14,221,0.06));border-radius:8px;margin-top:0.5rem; }
.dot-pulse { display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--primary);animation:pulse 1.4s ease-in-out infinite; }
.dot-pulse:nth-child(2) { animation-delay:0.2s; }
.dot-pulse:nth-child(3) { animation-delay:0.4s; }
@keyframes pulse { 0%,100%{opacity:0.2;transform:scale(0.7)} 50%{opacity:1;transform:scale(1.2)} }
</style>
"""

TOOL_NAMES = {
    "normalize_medical_term": "Normalize Medical Term",
    "get_lexical": "GraphQL: Get Lexical",
    "get_allowed_refinements": "GraphQL: Get Allowed Refinements",
    "get_narrower_sequential_refinements": "GraphQL: Narrower Sequential Refinements",
}

def is_normalize(tool_name):
    return "normalize" in tool_name.lower()

def fmt_tool_input(tool_name, args):
    if tool_name == "normalize_medical_term":
        return f'Normalizing <strong>\"{esc(args.get("input_term", ""))}\"</strong> in domain <strong>{esc(args.get("domain", "Problem"))}</strong>'
    elif tool_name == "get_lexical":
        return f'Looking up IMO lexical code <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    elif tool_name == "get_narrower_sequential_refinements":
        return f'Resolving narrower concept for <strong>{esc(args.get("imo_lexical_code", ""))}</strong> with refinements <strong>{esc(json.dumps(args.get("refinement_sequence", [])))}</strong>'
    elif tool_name == "get_allowed_refinements":
        return f'Getting allowed refinements for <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    return f'<pre>{esc(json.dumps(args, indent=2))}</pre>'

def fmt_tool_result_html(tool_name, result_str):
    try:
        data = json.loads(result_str) if isinstance(result_str, str) else result_str
    except (json.JSONDecodeError, TypeError):
        return f'<pre>{esc(str(result_str)[:500])}</pre>'
    
    html_out = ""
    if tool_name == "normalize_medical_term" and isinstance(data, dict) and data.get("results"):
        matches = data["results"][0].get("matches", []) if data["results"] else []
        term = data["results"][0].get("input_term", "") if data["results"] else ""
        html_out += f'<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">{len(matches)} result(s) for \"<strong>{esc(term)}</strong>\"</div>'
        html_out += '<div class="result-table-wrap"><table class="result-table"><thead><tr><th>IMO Description</th><th>Lexical Code</th><th>Score</th><th>ICD-10</th></tr></thead><tbody>'
        for m in matches[:5]:
            code = m.get("default_lexical_code", m.get("lexical_code", ""))
            icd = ", ".join(m.get("icd10_codes", []))
            html_out += f'<tr><td><strong>{esc(m.get("title", ""))}</strong></td><td>{esc(code)}</td><td>{m.get("score", 0):.3f}</td><td>{esc(icd)}</td></tr>'
        html_out += '</tbody></table></div>'
    elif tool_name == "get_lexical" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        html_out += f'<div style="font-size:0.83rem;color:var(--text-secondary);">Lexical: <strong>{esc(lex.get("title", ""))}</strong> (code: {esc(lex.get("code", ""))})</div>'
        allowed = lex.get("allowedRefinements", []) or []
        if allowed:
            groups = defaultdict(list)
            for ref in allowed:
                g = (ref.get("group") or {}).get("title", "Ungrouped")
                groups[g].append(ref)
            html_out += f'<div style="margin-top:0.4rem;"><strong>Allowed Refinement Groups ({len(groups)}):</strong></div>'
            for gname, refs in sorted(groups.items()):
                html_out += f'<div style="margin:0.3rem 0 0.1rem;padding:0.3rem 0.5rem;background:#f8fafc;border:1px solid var(--border);border-radius:6px;"><strong>{esc(gname)}</strong> <span style="font-size:0.7rem;background:var(--primary);color:#fff;padding:0.1rem 0.3rem;border-radius:999px;">{len(refs)}</span></div>'
                html_out += '<div class="result-table-wrap" style="max-height:180px;"><table class="result-table"><thead><tr><th>Refinement</th><th>Code</th></tr></thead><tbody>'
                for r in refs:
                    html_out += f'<tr><td>{esc(r.get("title", ""))}</td><td>{esc(r.get("code", ""))}</td></tr>'
                html_out += '</tbody></table></div>'
    elif tool_name == "get_narrower_sequential_refinements" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        narrower = lex.get("refinementNarrower", []) or []
        html_out += f'<div style="font-size:0.83rem;">Base: <strong>{esc(lex.get("title", ""))}</strong></div>'
        if narrower:
            html_out += f'<div><strong>Resolved ({len(narrower)}):</strong></div>'
            html_out += '<div class="result-table-wrap"><table class="result-table"><thead><tr><th>Title</th><th>Code</th><th>Applied Refinements</th></tr></thead><tbody>'
            for n in narrower:
                applied = ", ".join(r.get("title", "") for r in (n.get("appliedRefinements") or []))
                html_out += f'<tr><td><strong>{esc(n.get("title", ""))}</strong></td><td>{esc(n.get("code", ""))}</td><td>{esc(applied)}</td></tr>'
            html_out += '</tbody></table></div>'
        else:
            html_out += '<div><em>No narrower concepts resolved.</em></div>'
    else:
        html_out += f'<pre style="max-height:200px;overflow:auto;font-size:0.72rem;">{esc(json.dumps(data, indent=2)[:1000])}</pre>'
    
    raw_json = json.dumps(data, indent=2)
    html_out += f'<details class="raw-toggle"><summary>View Response JSON</summary><pre>{esc(raw_json[:3000])}</pre></details>'
    return html_out

def fmt_markdown_to_html(text):
    text = re.sub(r'\[SUMMARY\][\s\S]*?\[/SUMMARY\]', '', text).strip()
    lines = text.split('\n')
    result = []
    in_table = False
    table_rows = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and stripped.endswith('|'):
            if re.match(r'^\|[\s\-:|]+\|$', stripped):
                continue
            if not in_table:
                in_table = True
                table_rows = []
            table_rows.append([c.strip() for c in stripped.split('|')[1:-1]])
        else:
            if in_table:
                result.append(_build_table(table_rows))
                in_table = False
            result.append(stripped)
    if in_table:
        result.append(_build_table(table_rows))
    f = '\n'.join(result)
    f = re.sub(r'^#### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^## (.+)$', r'<h3>\1</h3>', f, flags=re.M)
    f = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', f)
    f = re.sub(r'\*(.+?)\*', r'<em>\1</em>', f)
    f = re.sub(r'`(.+?)`', r'<code>\1</code>', f)
    f = f.replace('\n\n', '</p><p>').replace('\n', '<br>')
    return f'<p>{f}</p>'

def _build_table(rows):
    if not rows:
        return ''
    html_t = '<table class="result-table" style="width:100%;margin:0.5rem 0;"><thead><tr>'
    for c in rows[0]:
        html_t += f'<th>{c}</th>'
    html_t += '</tr></thead><tbody>'
    for row in rows[1:]:
        html_t += '<tr>' + ''.join(f'<td>{c}</td>' for c in row) + '</tr>'
    html_t += '</tbody></table>'
    return html_t

def _streaming_dots(msg="Agent is working..."):
    return f'''<div class="streaming-indicator">
        <span class="dot-pulse"></span><span class="dot-pulse"></span><span class="dot-pulse"></span>
        <span style="margin-left:0.3rem;">{esc(msg)}</span>
    </div>'''

def render_agent_run(activity, final_text, label="Final Recommendation", streaming=False):
    """Render activity list into rich HTML. When streaming=True shows a live indicator."""
    normalize_n = sum(1 for a in activity if a["type"] == "tool_call" and is_normalize(a["tool"]))
    kg_n = sum(1 for a in activity if a["type"] == "tool_call" and not is_normalize(a["tool"]))
    count_parts = []
    if normalize_n: count_parts.append(f"{normalize_n} Normalize")
    if kg_n: count_parts.append(f"{kg_n} KG")
    count_label = " \u00b7 ".join(count_parts) if count_parts else "0 calls"

    norm_idx = 0
    kg_idx = 0
    steps_html = ""
    for a in activity:
        if a["type"] == "reasoning" and a.get("text", "").strip():
            steps_html += f'''\n            <div class="step step-reasoning">
                <div class="step-hdr"><span class="step-icon icon-reasoning">\U0001f9e0</span><span class="step-title">Agent Reasoning</span></div>
                <div class="reasoning-text">{fmt_markdown_to_html(a["text"])}</div>
            </div>'''
        elif a["type"] == "tool_call":
            tool = a["tool"]
            norm = is_normalize(tool)
            if norm:
                norm_idx += 1; idx = norm_idx
            else:
                kg_idx += 1; idx = kg_idx
            cls = "step-tool normalize" if norm else "step-tool"
            badge_cls = "badge-normalize" if norm else "badge-kg"
            badge_lbl = "Normalize" if norm else "KG Call"
            icon_cls = "icon-normalize" if norm else "icon-tool"
            icon = "\u2699" if norm else "\U0001f50d"
            display_name = TOOL_NAMES.get(tool, tool)
            detail = fmt_tool_input(tool, a.get("args", {}))
            steps_html += f'''\n            <div class="step {cls}">
                <span class="step-badge {badge_cls}">{badge_lbl} {idx}</span>
                <div class="step-hdr"><span class="step-icon {icon_cls}">{icon}</span><span class="step-title">{esc(display_name)}</span><span class="step-badge badge-done" style="position:static;margin-left:auto;">Done</span></div>
                <div class="step-detail">{detail}</div>
            </div>'''
        elif a["type"] == "tool_result":
            tool = a["tool"]
            display_name = TOOL_NAMES.get(tool, tool)
            result_html = fmt_tool_result_html(tool, a.get("result", ""))
            steps_html += f'''\n            <div class="step step-result">
                <div class="step-hdr"><span class="step-icon icon-result">\u2630</span><span class="step-title">Results from {esc(display_name)}</span></div>
                <div>{result_html}</div>
            </div>'''

    # Bottom section: final answer or streaming indicator
    if final_text:
        bottom = f'''<div class="final-answer">
            <div class="final-label">{esc(label)}</div>
            <div class="final-body">{fmt_markdown_to_html(final_text)}</div>
        </div>'''
    elif streaming:
        bottom = _streaming_dots("Agent is working...")
    else:
        bottom = ""

    # If no activity yet and streaming, just show the indicator
    if not activity and streaming:
        reasoning_section = _streaming_dots("Agent is thinking...")
    else:
        reasoning_section = f'''<div class="reasoning-steps">
            <div class="reasoning-hdr" onclick="var b=this.parentElement.querySelector('.reasoning-body');b.style.display=b.style.display==='none'?'flex':'none'">
                <h4>Agent Reasoning &amp; API Calls</h4>
                <span class="step-count">{count_label}</span>
                <span class="toggle-icon">\u25bc</span>
            </div>
            <div class="reasoning-body">{steps_html}</div>
        </div>'''

    return f'''{AGENT_CSS}
    <div class="agent-panel">
        <div class="msg-container">
            <div class="msg-avatar">\U0001f916</div>
            <div class="msg-content">
                <div class="msg-header">
                    <span class="msg-sender">Diagnosis Specificity Agent</span>
                </div>
                {reasoning_section}
                {bottom}
            </div>
        </div>
    </div>'''

print("\u2713  HTML renderer + streaming engine ready")

✓  HTML renderer + streaming engine ready


## 3. Agent + Streaming Runner

Creates the ReAct agent and provides `stream_agent()` — streams via
`agent.stream(stream_mode='values')` and live-updates the HTML display
after every agent/tool step.

In [44]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools=DS_CONDITIONS_TOOLS, prompt=SYSTEM_PROMPT)

def _extract_text(content):
    """Extract text from msg.content (str or list of content blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return str(content) if content else ""

def _build_activity(messages, skip_count=0):
    """Parse messages into (activity_list, final_text). Skip first skip_count messages."""
    activity = []
    final_text = ""
    for msg in messages[skip_count:]:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role == 'ai':
            text = _extract_text(msg.content)
            if text.strip():
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    activity.append({"type": "reasoning", "text": text})
                else:
                    final_text = text
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    activity.append({"type": "tool_call", "tool": tc["name"], "args": tc.get("args", {})})
        elif role == 'tool':
            content = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
            activity.append({"type": "tool_result", "tool": msg.name, "result": content})
    return activity, final_text

def stream_agent(input_messages, display_id, label="Final Recommendation",
                 skip_count=0, recursion_limit=100):
    """Stream agent execution with live HTML updates.
    Returns (activity, final_text, all_messages).
    """
    # Show initial loading
    update_display(HTML(render_agent_run([], "", label=label, streaming=True)), display_id=display_id)

    last_state = None
    for state in agent.stream(
        {"messages": input_messages},
        stream_mode="values",
        config={"recursion_limit": recursion_limit},
    ):
        last_state = state
        activity, final_text = _build_activity(state["messages"], skip_count=skip_count)
        html = render_agent_run(activity, final_text, label=label, streaming=not bool(final_text))
        update_display(HTML(html), display_id=display_id)

    all_messages = last_state["messages"] if last_state else input_messages
    activity, final_text = _build_activity(all_messages, skip_count=skip_count)
    # Final render (streaming=False)
    update_display(HTML(render_agent_run(activity, final_text, label=label, streaming=False)), display_id=display_id)
    return activity, final_text, all_messages

print("\u2713  Agent + streaming runner ready")

✓  Agent + streaming runner ready


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_13168\3376935075.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=DS_CONDITIONS_TOOLS, prompt=SYSTEM_PROMPT)


## 4. Clinical Note

In [45]:
CLINICAL_NOTE = """65-year-old female with essential hypertension and chronic kidney disease stage 3. BP today 148/92. eGFR 45. Proteinuria present on urine dipstick. Assessment: Hypertensive chronic kidney disease with stage 3 CKD. Increase lisinopril, recheck labs in 3 months."""

user_html = f'''
{AGENT_CSS}
<div class="agent-panel">
    <div class="msg-container msg-user">
        <div class="msg-avatar">\U0001f464</div>
        <div class="msg-content">
            <div class="msg-header">
                <span class="msg-sender">You</span>
            </div>
            <div class="clinical-note">
                <div class="clinical-note-hdr">
                    <span style="font-size:1.3rem;">\U0001f4cb</span>
                    <span class="clinical-note-title" style="color:#fff;">Clinical Note</span>
                </div>
                <div class="clinical-note-body" style="color:#fff;">{esc(CLINICAL_NOTE)}</div>
            </div>
        </div>
    </div>
</div>
'''
display(HTML(user_html))

## 5. Run Agent — Extraction → Selection → Refinement (Streaming)

Single cell that runs the full agent flow:
1. **Phase 1** — Streams extraction, shows clinical note with highlighted diagnoses
2. **Selection** — Pick which diagnosis to refine via `input()` prompt
3. **Phase 2-4** — Streams refinement with all tool calls and reasoning rendered live

In [46]:
import threading, time
from http.server import HTTPServer, BaseHTTPRequestHandler
import urllib.parse

# ═══════════════════════════════════════════════════════
# PHASE 1 — Stream extraction
# ═══════════════════════════════════════════════════════
p1_stream_id = f"phase1_{uuid.uuid4().hex[:6]}"
display(HTML(render_agent_run([], "", label="Extracted Diagnoses", streaming=True)), display_id=p1_stream_id)

p1_activity, p1_final, phase1_messages = stream_agent(
    [{"role": "user", "content": CLINICAL_NOTE}],
    display_id=p1_stream_id,
    label="Extracted Diagnoses",
    skip_count=1,
    recursion_limit=50,
)

extraction_text = p1_final

# --- Parse numbered list ---
EXTRACTED_DIAGNOSES = []
for line in extraction_text.split('\n'):
    m = re.match(r'^\d+[\.)\-]\s*\*{0,2}(.+?)\*{0,2}\s*$', line.strip())
    if m:
        dx = m.group(1).strip()
        if dx and not any(kw in dx.lower() for kw in ['extract', 'refine', 'select', 'which']):
            EXTRACTED_DIAGNOSES.append(dx)

if not EXTRACTED_DIAGNOSES:
    EXTRACTED_DIAGNOSES = ["Hypertension", "Chronic kidney disease"]

# --- Highlight clinical note ---
def highlight_note(note, diagnoses):
    h = esc(note)
    for dx in diagnoses:
        pattern = re.compile(re.escape(esc(dx)), re.IGNORECASE)
        h = pattern.sub(lambda m: f'<span class="hl">{m.group()}</span>', h)
    return h

highlighted = highlight_note(CLINICAL_NOTE, EXTRACTED_DIAGNOSES)

# ═══════════════════════════════════════════════════════
# SELECTION — Local HTTP bridge for HTML → Python
# ═══════════════════════════════════════════════════════
_sel_holder = [None]
_sel_event = threading.Event()

class _SelHandler(BaseHTTPRequestHandler):
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, OPTIONS')
        self.end_headers()
    def do_GET(self):
        qs = urllib.parse.parse_qs(urllib.parse.urlparse(self.path).query)
        if 'dx' in qs:
            _sel_holder[0] = qs['dx'][0]
            _sel_event.set()
        self.send_response(200)
        self.send_header('Content-Type', 'text/plain')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(b'ok')
    def log_message(self, *args): pass

_srv = HTTPServer(('127.0.0.1', 0), _SelHandler)
_port = _srv.server_address[1]

def _serve_loop():
    while not _sel_event.is_set():
        _srv.handle_request()

threading.Thread(target=_serve_loop, daemon=True).start()

# --- Build radio items ---
radio_items = ""
for i, dx in enumerate(EXTRACTED_DIAGNOSES):
    checked = 'checked' if i == 0 else ''
    radio_items += (
        '<label class="dx-select-item" onclick="this.querySelector(\'input\').checked=true;'
        "document.querySelectorAll('.dx-select-item').forEach(l=>{l.style.borderColor='var(--border)';l.style.background='var(--bg)'});"
        "this.style.borderColor='var(--primary)';this.style.background='rgba(87,3,158,0.06)'\">"
        f'<input type="radio" name="dx_radio" value="{esc(dx)}" {checked}>'
        f'<span style="font-size:1.05rem;">{i+1}. {esc(dx)}</span>'
        '</label>'
    )

default_dx = esc(EXTRACTED_DIAGNOSES[0])
js_block = """
<script>
function handleRefine() {
    var sel = document.querySelector('input[name="dx_radio"]:checked');
    if (!sel) return;
    var val = sel.value;
    var st = document.getElementById('refine_status');
    st.classList.add('show');
    document.getElementById('refine_status_text').innerHTML = 'Sending: <b>' + val + '</b>...';
    fetch('http://127.0.0.1:""" + str(_port) + """/?dx=' + encodeURIComponent(val))
        .then(function() {
            document.getElementById('refine_status_text').innerHTML = '✅ Selected: <b>' + val + '</b> — Agent is running Phase 2-4...';
        })
        .catch(function(e) {
            document.getElementById('refine_status_text').innerHTML = '⚠ Fetch blocked. Sending default...';
            fetch('http://127.0.0.1:""" + str(_port) + """/?dx=""" + default_dx + """');
        });
}
</script>
"""

selection_html = (
    AGENT_CSS
    + '<div class="agent-panel"><div class="msg-container">'
    + '<div class="msg-avatar" style="background:var(--accent);">\U0001f916</div>'
    + '<div class="msg-content">'
    + '<div class="msg-header"><span class="msg-sender">Diagnosis Specificity Agent</span>'
    + '<span class="msg-time">Phase 1 — Extraction Complete</span></div>'
    + '<div class="clinical-note"><div class="clinical-note-hdr">'
    + '<span style="font-size:1.3rem;">\U0001f4cb</span>'
    + '<span class="clinical-note-title">Clinical Note with Extracted Problems</span></div>'
    + f'<div class="clinical-note-body">{highlighted}</div></div>'
    + '<div style="margin:1rem 0 0.5rem;font-weight:600;">Select a diagnosis to refine:</div>'
    + f'<div id="dx_radio_group">{radio_items}</div>'
    + '<button class="refine-btn" onclick="handleRefine()" style="margin-top:1rem;">▶ Refine Selected Diagnosis</button>'
    + '<div id="refine_status" class="status-banner">'
    + '<span style="font-size:1.1rem;">✅</span>'
    + '<span id="refine_status_text" style="font-weight:600;color:#065f46;margin-left:0.4rem;"></span>'
    + '</div></div></div></div>'
    + js_block
)
display(HTML(selection_html))

# --- Wait for selection ---
print("⏳ Waiting for selection... Click 'Refine Selected Diagnosis' above.")
_sel_event.wait(timeout=300)
_srv.server_close()

if _sel_holder[0]:
    selected = _sel_holder[0]
else:
    selected = EXTRACTED_DIAGNOSES[0]
    print(f"  Timeout. Using default: {selected}")

user_msg = f"Refine: {selected}"
print(f"\n→ Refining: {selected}\n")

# ═══════════════════════════════════════════════════════
# PHASE 2-4 — Stream refinement with full activity
# ═══════════════════════════════════════════════════════
user_html2 = (
    AGENT_CSS
    + '<div class="agent-panel"><div class="msg-container msg-user">'
    + '<div class="msg-avatar">\U0001f464</div>'
    + '<div class="msg-content">'
    + '<div class="msg-header"><span class="msg-sender">You</span></div>'
    + f'<div>{esc(user_msg)}</div>'
    + '</div></div></div>'
)
display(HTML(user_html2))

messages_for_refine = phase1_messages + [{"role": "user", "content": user_msg}]
skip_n = len(phase1_messages) + 1

p2_stream_id = f"phase2_{uuid.uuid4().hex[:6]}"
display(HTML(render_agent_run([], "", label="Final Recommendation", streaming=True)), display_id=p2_stream_id)

p2_activity, p2_final, all_messages = stream_agent(
    messages_for_refine,
    display_id=p2_stream_id,
    label="Final Recommendation",
    skip_count=skip_n,
    recursion_limit=100,
)

tc_count = sum(1 for a in p2_activity if a['type'] == 'tool_call')
rs_count = sum(1 for a in p2_activity if a['type'] == 'reasoning')
print(f"✓ Complete. {tc_count} tool calls, {rs_count} reasoning steps.")

ClientError: An error occurred (ExpiredTokenException) when calling the Converse operation: The security token included in the request is expired